### 1. Emlakjet sitesi ilan bilgilerinden ayrıntılı bilgi çekme.

In [3]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd

In [4]:
# ilk versiyon olarak direk ilandan veri çekicez. daha sonrasında şehire göre ilanları gezerken her ilan için ilanın sayfasına girip oradan ayrıntılı ilan bilgilerini çekicek.
url = "https://www.emlakjet.com/ilan/derya-park-evlerinde-genis-bahce-kullanimli-villa-tadinda-ful-esyali-kiralik-90m2-11-daire-17927718"

response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# tag ve attribute parametrelerini verince ilgili verileri çekiyor.
def data_scraping(tag:str, attr_value:str) -> dict:
    
    data_dict = {} 
    container = soup.find(f"{tag}", class_ = f"{attr_value}")
    
    if container:
        items = container.find_all("li")
        
        for li in items:
            key_span = li.find("span", class_="styles_key__wX_g4")
            value_span = li.find("span", class_="styles_value__xmNV3")

            if key_span and value_span:
                key = key_span.get_text(strip=True)
                value = value_span.get_text(strip=True)
                data_dict[key] = value
    
    return data_dict

a = data_scraping("div", "styles_wrapper__WDIOr")
# a değişkeni bir ilan sayfasındaki ilan bilgilerini dict türünde getirir. key değerleri bilgilerin isimleri (net metrekare, bina yaşı şeklinde), value değerleri ise verileri veriyor.

In [5]:
a

{'İlan Numarası': '17927718',
 'İlan Oluşturma Tarihi': '21 Ağustos 2025',
 'İlan Güncelleme Tarihi': '21 Ağustos 2025',
 'Türü': 'Konut',
 'Kategorisi': 'Kiralık',
 'Tipi': 'Daire',
 'Net Metrekare': '85 m²',
 'Brüt Metrekare': '90 m²',
 'Oda Sayısı': '1+1',
 'Binanın Yaşı': '16-20',
 'Bulunduğu Kat': 'Bahçe Katı',
 'Binanın Kat Sayısı': '4',
 'Isıtma Tipi': 'Kombi Doğalgaz',
 'Kullanım Durumu': 'Mülk Sahibi Oturuyor',
 'Aidat': '2200 TL',
 'Tapu Durumu': 'Kat Mülkiyeti',
 'Site İçerisinde': 'Evet',
 'Depozito': '60000 TL',
 'Banyo Sayısı': '1',
 'Balkon Durumu': 'Var',
 'Fiyat Durumu': 'Genel Fiyat',
 'Ada': '486',
 'Parsel': '1'}

In [6]:
len(a)

23

### 2. Sayfalarda dolaşma

#### 2.1 Şehirlerin kaç sayfa ilanı olduğunu buluyor sonuçları ise dictionary de saklıyor. 

In [ ]:
cities = [
    "adana", "adiyaman", "afyonkarahisar", "agri", "aksaray", "amasya", "ankara", "antalya",
    "ardahan", "artvin", "aydin", "balikesir", "bartin", "batman", "bayburt", "bilecik",
    "bingol", "bitlis", "bolu", "burdur", "bursa", "canakkale", "cankiri", "corum",
    "denizli", "diyarbakir", "duzce", "edirne", "elazig", "erzincan", "erzurum",
    "eskisehir", "gaziantep", "giresun", "gumushane", "hakkari", "hatay", "igdir",
    "isparta", "mersin", "istanbul", "izmir", "kahramanmaras", "karabuk", "karaman",
    "kars", "kastamonu", "kayseri", "kilis", "kirikkale", "kirklareli", "kirsehir",
    "kocaeli", "konya", "kutahya", "malatya", "manisa", "mardin", "mugla", "mus",
    "nevsehir", "nigde", "ordu", "osmaniye", "rize", "sakarya", "samsun", "sanliurfa",
    "siirt", "sinop", "sivas", "tekirdag", "tokat", "trabzon", "tunceli", "usak",
    "van", "yalova", "yozgat", "zonguldak", "sirnak"
]
cities_pages = {}

for city in cities:
    url = f"https://www.emlakjet.com/kiralik-konut/{city}"

    print(f"Sayfa {city} -> {url}")

    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

   
    page_numbers = []
    for li in soup.select("ul.styles_list__zqOeW li"):
        text = li.get_text(strip = True) # li içindeki text'i alıyoruz ve boşlukları temizliyoruz
        if text.isdigit():# eğer text tamamen sayılardan oluşuyorsa
            page_numbers.append(int(text)) # int'e çevirip listeye ekliyoruz

    if page_numbers:
        total_pages = max(page_numbers)
        print("toplam sayfa sayısı",total_pages)
    else:
        total_pages = 1
        print("Toplam sayfa sayısı",total_pages)

    cities_pages[city] = total_pages

In [10]:
print("\nTüm şehirlerin sayfa sayıları:")
print(cities_pages)


Tüm şehirlerin sayfa sayıları:
{'adana': 47, 'adiyaman': 1, 'afyonkarahisar': 7, 'agri': 1, 'aksaray': 4, 'amasya': 6, 'ankara': 50, 'antalya': 50, 'ardahan': 1, 'artvin': 1, 'aydin': 36, 'balikesir': 40, 'bartin': 3, 'batman': 5, 'bayburt': 1, 'bilecik': 3, 'bingol': 1, 'bitlis': 1, 'bolu': 3, 'burdur': 4, 'bursa': 50, 'canakkale': 10, 'cankiri': 1, 'corum': 3, 'denizli': 32, 'diyarbakir': 6, 'duzce': 8, 'edirne': 8, 'elazig': 3, 'erzincan': 1, 'erzurum': 6, 'eskisehir': 30, 'gaziantep': 16, 'giresun': 6, 'gumushane': 1, 'hakkari': 1, 'hatay': 4, 'igdir': 1, 'isparta': 12, 'mersin': 39, 'istanbul': 50, 'izmir': 50, 'kahramanmaras': 3, 'karabuk': 2, 'karaman': 1, 'kars': 4, 'kastamonu': 3, 'kayseri': 15, 'kilis': 5, 'kirikkale': 2, 'kirklareli': 9, 'kirsehir': 1, 'kocaeli': 34, 'konya': 13, 'kutahya': 6, 'malatya': 5, 'manisa': 23, 'mardin': 6, 'mugla': 50, 'mus': 1, 'nevsehir': 2, 'nigde': 2, 'ordu': 14, 'osmaniye': 2, 'rize': 3, 'sakarya': 25, 'samsun': 41, 'sanliurfa': 17, 'siirt':

#### 2.2 Her şehirlerdeki sayfaları dolaşmak 

##### URL oluşturma mantığı : dolaşılıcak tüm url leri bulur

In [17]:
base_url = "https://www.emlakjet.com/kiralik-daire/"
total = 0
for city, total_pages in cities_pages.items():
    
    for page in range(1, total_pages + 1):
        if page == 1:
            url = f"{base_url}{city}"
        else:
            url = f"{base_url}{city}?sayfa={page}"
        total +=1
        print(url)
print(total)

https://www.emlakjet.com/kiralik-daire/adana
https://www.emlakjet.com/kiralik-daire/adana?sayfa=2
https://www.emlakjet.com/kiralik-daire/adana?sayfa=3
https://www.emlakjet.com/kiralik-daire/adana?sayfa=4
https://www.emlakjet.com/kiralik-daire/adana?sayfa=5
https://www.emlakjet.com/kiralik-daire/adana?sayfa=6
https://www.emlakjet.com/kiralik-daire/adana?sayfa=7
https://www.emlakjet.com/kiralik-daire/adana?sayfa=8
https://www.emlakjet.com/kiralik-daire/adana?sayfa=9
https://www.emlakjet.com/kiralik-daire/adana?sayfa=10
https://www.emlakjet.com/kiralik-daire/adana?sayfa=11
https://www.emlakjet.com/kiralik-daire/adana?sayfa=12
https://www.emlakjet.com/kiralik-daire/adana?sayfa=13
https://www.emlakjet.com/kiralik-daire/adana?sayfa=14
https://www.emlakjet.com/kiralik-daire/adana?sayfa=15
https://www.emlakjet.com/kiralik-daire/adana?sayfa=16
https://www.emlakjet.com/kiralik-daire/adana?sayfa=17
https://www.emlakjet.com/kiralik-daire/adana?sayfa=18
https://www.emlakjet.com/kiralik-daire/adana?

URL leri bir listede sakla

In [ ]:
base_url = "https://www.emlakjet.com/kiralik-daire/"
url_list = []
for city, total_pages in cities_pages.items():
    
    for page in range(1, total_pages + 1):
        if page == 1:
            url = f"{base_url}{city}"
        else:
            url = f"{base_url}{city}?sayfa={page}"
        url_list.append(url)
        
print(url_list)

['https://www.emlakjet.com/kiralik-daire/adana', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=2', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=3', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=4', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=5', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=6', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=7', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=8', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=9', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=10', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=11', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=12', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=13', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=14', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=15', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=16', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=17', 'https://www.emlakjet.com/kiralik-daire/adana?s

### 3. İşi bitir

In [ ]:
def get_ads_from_page(url: str) -> list:
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    
    ads = []
    for a_tag in soup.select("a[href*='/ilan/']"):
        link = a_tag['href']
        if link.startswith("/ilan/"):
            link = "https://www.emlakjet.com" + link
        ads.append(link)
    
    return list(set(ads))  # tekrar edenleri temizle
